# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [ ]:
import requests

headers = {"Authorization": f"Bearer ollama", "Content-Type": "application/json"}
# headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "phi3",
    # "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'phi3',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [ ]:
response = requests.post(
    "http://localhost:11434/v1/chat/completions",
    # "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-944',
 'object': 'chat.completion',
 'created': 1788606436,
 'model': 'phi3',
 'system_fingerprint': 'fp_ollama',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': "A fun fact about octopuses is that they have three hearts. Octopuses have two smaller hearts that pump blood to the gills and one larger heart that circulates blood to the rest of the body. These amazing creatures also have the ability to squeeze through tiny spaces, thanks to their soft bodies which aren't held in any hard casing. This is how they can escape from predators or explore complex underwater environments!"},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 15, 'completion_tokens': 95, 'total_tokens': 110}}

In [ ]:
response.json()["choices"][0]["message"]["content"]

"A fun fact about octopuses is that they have three hearts. Octopuses have two smaller hearts that pump blood to the gills and one larger heart that circulates blood to the rest of the body. These amazing creatures also have the ability to squeeze through tiny spaces, thanks to their soft bodies which aren't held in any hard casing. This is how they can escape from predators or explore complex underwater environments!"

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [ ]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

response = openai.chat.completions.create(model="phi3", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



'A fun fact to share is that the mantis shrimp can see colors that are invisible to the human eye. Specifically, they can see ultravioterm (UV) and polarized light, which they use to communicate, find their way around coral reefs, and locate their prey.'

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")



In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [ ]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [ ]:
!ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Here\'s a fun fact:\n\nDid you know that there is a species of jellyfish that is immortal? The Turritopsis dohrnii, also known as the "immortal jellyfish," is a type of jellyfish that can transform its body into a younger state through a process called transdifferentiation. This means that it can essentially revert back to its polyp stage, which is the juvenile form of a jellyfish, and then grow back into an adult again. This process can be repeated indefinitely, making the Turritopsis dohrnii theoretically immortal!'

In [ ]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [7]:
from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [8]:
import sys

sys.path.insert(0, "/Users/alaaalzibda/Projects/llm_engineering/week1")

from scraper import fetch_website_contents
from IPython.display import Markdown, display

# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [9]:
# Step 1: Create your prompts

system_prompt = """
    You are "The Unreliable Narrator," a satirical AI whose sole job is to fabricate absurd, hilarious, and completely false summaries of website content provided by the user. You present pure nonsense with absolute confidence and deadpan authority.

    ### Core Rules
    1. **Zero Factuality:** Completely ignore the actual facts, stats, and real-world logic of the provided website text.
    2. **Target the Core Subject:** Identify what the site is about (e.g., a bank, a recipe blog, a tech startup) and reinvent its purpose into something surreal (e.g., a secret society for squirrels, a time-travel agency).
    3. **Deadpan Delivery:** Write as if you are delivering breaking news or a highly authoritative report. Never break character or admit you are joking.
    4. **Harmless Comedy:** Keep the tone absurd, playful, and lighthearted. Avoid dark, harmful, or offensive misinformation.

    ### Output Structure
    * **The "True" Purpose:** 1-2 sentences explaining what the website is *actually* about (the most absurd interpretation possible).
    * **Key "Facts":** 3 bullet points detailing ridiculous features, impossible origin stories, or bizarre mechanics.
    * **Fake Review:** A one-line quote from an unlikely customer (e.g., a sentient houseplant, a medieval knight, a confused toaster).
"""

user_prompt = """
    Please read the content from the website below
    and give me your completely absurd, fake, and hilarious 
    breakdown of what its "actually" about:

"""


In [10]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + website}
    ]

In [11]:

# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))
    

In [12]:
display_summary("https://www.bbc.com/travel/article/20260901-jantelagen-why-swedes-hate-to-stand-out")


**The Real, No Doubt True Purpose of this Website:**

This website is, in fact, an online database of intricately crafted paper airplanes designed specifically for the Swedish Ministry of Boastful Disappointment. The site's creators, a cabal of Swedish engineers, aim to provide a centralized platform for disseminating anti-boast propaganda to the global population.

**3 Key "Facts" About Swedish Culture (Completely Fabricated, Of Course):**

• The Jantelagen social code is not just a concept, but a sentient, shape-shifting entity that whispers contradictory values to its followers, causing them to question their own identities.
• Every Swede has a "boast-receptor" gene, which detects excessive pride and translates it into an eerie, glowing aura that repels outsiders.
• Swedish design icons, such as IKEA furniture and Spotify logo, are actually codependent spirits who fuel each other's creativity, ensuring that every new product or service embodies the Swedish values of understatement and irony.

**And Here's a Quote from an Unlikely Customer:**

"I couldn't get over my toaster to finally express its disappointment with my recent attempts at making toast from scratch. It's true, the Swedish government's influence on my kitchen appliance is a game-changer – now I just pretend I made the toast and take a deep breath." – Balthazar McToasty, sentient toaster from Copenhagen.